<a href="https://colab.research.google.com/github/Joell-Alex1/Chess-Comment-Toxicity-Domain-Classification/blob/main/Chess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import string
import re   # you forgot this import

sentences = {
    "toxic": [
        "You are using     Stockfish!    ",
        "Nice !!//?game bro!! running"
    ]
}

df = pd.DataFrame(sentences)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    translator = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
    text = text.translate(translator)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)


    return text

# Apply function to each row
df["cleaned"] = df["toxic"].apply(clean_text)

print(df)




                              toxic                  cleaned
0  You are using     Stockfish!      you are using stockfish
1      Nice !!//?game bro!! running    nice game bro running


In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize


df["tokens"] = df["cleaned"].apply(word_tokenize)

print(df["tokens"])

0    [you, are, using, stockfish]
1      [nice, game, bro, running]
Name: tokens, dtype: object


Remove the STOPWORDS


In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
def remove_stopwords(tokens):

  filtered=[]
  for words in tokens:
    if words not in stop_words:
      filtered.append(words)
  return filtered


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
# lambda parameter: expression
# list comp is expression iteration if/else
df["tokens_no_stop"] = df["tokens"].apply(
    lambda words: [word for word in words if word not in stop_words]
)
print(df["tokens_no_stop"])

0            [using, stockfish]
1    [nice, game, bro, running]
Name: tokens_no_stop, dtype: object


In [ ]:
nltk.download("wordnet")

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

df["tokens_lemma"] = df["tokens_no_stop"].apply(
    lambda words: [lemmatizer.lemmatize(word, pos="v") for word in words]
)
df["final_text"] = df["tokens_lemma"].apply(lambda words: " ".join(words))
print(df)

                              toxic                  cleaned  \
0  You are using     Stockfish!      you are using stockfish   
1      Nice !!//?game bro!! running    nice game bro running   

                         tokens              tokens_no_stop  \
0  [you, are, using, stockfish]          [using, stockfish]   
1    [nice, game, bro, running]  [nice, game, bro, running]   

             tokens_lemma         final_text  
0        [use, stockfish]      use stockfish  
1  [nice, game, bro, run]  nice game bro run  


In [ ]:
!pip install youtube-comment-downloader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 9.3 MB/s eta 0:00:00


In [ ]:
from itertools import islice
from youtube_comment_downloader import *
downloader = YoutubeCommentDownloader()
comments_list=[]
comments = downloader.get_comments_from_url('https://www.youtube.com/watch?v=aDUmS_MJceU', sort_by=SORT_BY_POPULAR)
for comment in islice(comments, 100):

    print(clean_text(comment['text']))
    comments_list.append(clean_text(comment['text']))


df = pd.DataFrame(comments_list, columns=["toxic"])

# Run full pipeline on scraped data
df["cleaned"] = df["toxic"].apply(clean_text)
df["tokens"] = df["cleaned"].apply(word_tokenize)
df["tokens_no_stop"] = df["tokens"].apply(remove_stopwords)
df["tokens_lemma"] = df["tokens_no_stop"].apply(
    lambda words: [lemmatizer.lemmatize(word, pos="v") for word in words]
)
df["final_text"] = df["tokens_lemma"].apply(lambda words: " ".join(words))
print(df)

hi i m levy welcome to the chess world if you haven t played you should it s quite a fun game highly recommend my beginner videos this video was recorded the first day the drama happened so it wasn t as detailed as i wanted i made an updated more detailed video of the scandal here
heres my take im betting magnus and his inner circle have had suspicions since the ftx crypto cup possibly before specifically his game against hans the cheating wouldve involved the leak of prep not computer assistance which would be nearly impossible to get away with he decided to test this theory this time by preparing an obscure line he had never played before and his suspicions were confirmed he subsequently withdrew from the tournament not out of saltiness from losing to hans but because he now knew with near certainty that there was a leak within this camp and whos to say that the leaker hadnt leaked to others as well everyone is focusing on hans in all of this but to me the drastic reaction by magnus 

In [ ]:
!pip install detoxify

In [ ]:
from detoxify import Detoxify
results = Detoxify('original').predict(df["toxic"].tolist())

Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /root/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:01<00:00, 221MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
print(df)

                                                toxic  \
0   hi i m levy welcome to the chess world if you ...   
1   heres my take im betting magnus and his inner ...   
2   fun fact in a norwegian podcast magnus talked ...   
3   i bet someone cheated by stream sniping to see...   
4   clearly the engine is inside the pieces becaus...   
..                                                ...   
95  the funny part is none of that has anything to...   
96  yeah its either that or magnus is very very bu...   
97                                   you convinced me   
98  wow haven t thought of nor encountered this po...   
99  a large conspiracy with someone from magnus ca...   

                                              cleaned  \
0   hi i m levy welcome to the chess world if you ...   
1   heres my take im betting magnus and his inner ...   
2   fun fact in a norwegian podcast magnus talked ...   
3   i bet someone cheated by stream sniping to see...   
4   clearly the engine is insi

In [ ]:
df["tokens_no_stop"] = df["toxic"].apply(remove_stopwords)

In [ ]:
df["toxicity"] = results["toxicity"]
df["severe_toxicity"] = results["severe_toxicity"]
df["obscene"] = results["obscene"]
df["threat"] = results["threat"]
df["insult"] = results["insult"]
df["identity_attack"] = results["identity_attack"]
print(df)



                                                toxic  \
0   hi i m levy welcome to the chess world if you ...   
1   heres my take im betting magnus and his inner ...   
2   fun fact in a norwegian podcast magnus talked ...   
3   i bet someone cheated by stream sniping to see...   
4   clearly the engine is inside the pieces becaus...   
..                                                ...   
95  the funny part is none of that has anything to...   
96  yeah its either that or magnus is very very bu...   
97                                   you convinced me   
98  wow haven t thought of nor encountered this po...   
99  a large conspiracy with someone from magnus ca...   

                                              cleaned  \
0   hi i m levy welcome to the chess world if you ...   
1   heres my take im betting magnus and his inner ...   
2   fun fact in a norwegian podcast magnus talked ...   
3   i bet someone cheated by stream sniping to see...   
4   clearly the engine is insi

In [ ]:

chess_terms = {"engine", "stockfish", "elo", "niemann", "hans", "magnus", "carlsen", "prep", "gm"}

accusation_terms = {"using", "cheater", "caught", "banned", "reported", "sus", "suspicious",
                    "smurf", "sandbag", "cheat", "cheats", "leaked", "assistance", "traitor", "bot"}

direct_toxic = {"ez", "ezez", "patzer", "noob", "trash", "donkey", "woodpusher", "duffer", "trashcan", "zzz"}
def classify_comment(comment):
    has_chess_term = any(word in comment for word in chess_terms)
    has_accusation = any(word in comment for word in accusation_terms)
    toxic = any(re.search(r"\b" + word + r"\b", comment) for word in direct_toxic)

    if toxic:
        return "direct_toxic"
    if has_chess_term and has_accusation:
        return "cheating_accusation"
    elif has_chess_term:
        return "chess_mention"
    else:
        return "normal"

df["chess_flag"] = df["final_text"].apply(classify_comment)
print(df["chess_flag"].value_counts())

print(df)

chess_flag
normal                 59
chess_mention          25
cheating_accusation    16
Name: count, dtype: int64
                                                toxic  \
0   hi i m levy welcome to the chess world if you ...   
1   heres my take im betting magnus and his inner ...   
2   fun fact in a norwegian podcast magnus talked ...   
3   i bet someone cheated by stream sniping to see...   
4   clearly the engine is inside the pieces becaus...   
..                                                ...   
95  the funny part is none of that has anything to...   
96  yeah its either that or magnus is very very bu...   
97                                   you convinced me   
98  wow haven t thought of nor encountered this po...   
99  a large conspiracy with someone from magnus ca...   

                                              cleaned  \
0   hi i m levy welcome to the chess world if you ...   
1   heres my take im betting magnus and his inner ...   
2   fun fact in a norwegian p

In [ ]:
print(df["chess_flag"].value_counts())


chess_flag
normal                 59
chess_mention          25
cheating_accusation    16
Name: count, dtype: int64


In [ ]:
# from sklearn.metrics import classification_report

# labeled_df = pd.read_csv("chess_comments_labeled.csv")
# print(classification_report(labeled_df["manual_label"], df["chess_flag"]))

In [ ]:
# # Detoxify flags anything above 0.5 as toxic
# labeled_df["detoxify_flag"] = labeled_df["toxicity"].apply(
#     lambda x: "cheating_accusation" if x > 0.5 else "normal"
# )
# print(classification_report(labeled_df["manual_label"], labeled_df["detoxify_flag"]))

In [ ]:
df.to_csv("chess_comments.csv", index=False)